# 🧠 UEBA Lab — Notebook 03: LLM Reasoning on PCAI

This notebook is the **reasoning and alert-generation layer** of the UEBA pipeline.

It takes the structured anomaly outputs from Notebook 02 and transforms them into:

- analyst-readable SOC alert summaries,
- threat hypotheses,
- false-positive assessments,
- recommended next actions,
- structured JSON and CSV alert artifacts.

The design principle is important:

- **ML detects anomalous behavior**,
- **the LLM interprets and explains it**.

That separation keeps the workflow sane. The LLM is being used as a triage copilot, not as a magical anomaly detector in a very expensive hat.

## 🧭 End-to-end notebook flow

The notebook implements the following reasoning pipeline:

```text
flagged_users.csv + risk_scores.csv
              │
              ├── Load flagged cases and optional history
              │
              ├── Configure PCAI endpoint + model
              │
              ├── Test LLM connectivity
              │
              ├── Build structured evidence context per user
              │
              ├── Build reasoning prompt
              │
              ├── Call LLM for JSON alert generation
              │
              ├── Parse / validate JSON response
              │
              ├── Batch-generate alerts for flagged users
              │
              ├── Analyze quality + false-positive filtering
              │
              └── Save alerts.json + alerts_summary.csv
```

The key engineering pattern here is:

- **context building** → **controlled prompting** → **strict JSON parsing**.

That is exactly the right order for a production-minded LLM workflow.

## 🏗️ Architecture overview

This notebook sits after the anomaly detection stack.

| **Layer** | **Produced by** | **Consumed here as** |
|---|---|---|
| Daily behavior features | Notebook 01 | already collapsed into scores upstream |
| ML anomaly scores | Notebook 02 | structured evidence |
| SHAP top-5 contributors | Notebook 02 | explanation context |
| Peak flagged users | Notebook 02 | primary alert candidates |
| Risk score history | Notebook 02 | optional trend context |
| PCAI-hosted LLM | external endpoint | reasoning / alert synthesis engine |

And the high-level architecture is:

```text
Notebook 02 outputs
   ├── flagged_users.csv
   └── risk_scores.csv
            │
            ▼
   Evidence Context Builder
            │
            ▼
   Prompt Builder
            │
            ▼
   PCAI Chat Completion API
            │
            ▼
   JSON Parser / Validator
            │
            ▼
   Analyst-ready alerts
   ├── alerts.json
   └── alerts_summary.csv
```

## 🎯 Notebook goals

For a technical developer, the main things to understand in this notebook are:

1. **how evidence is structured before being sent to the LLM**,  
2. **how the JSON schema is enforced**,  
3. **how to safely parse and validate responses**,  
4. **how to distinguish actionable alerts from likely false positives**.

This notebook is less about model training and more about **LLM systems integration** and **analyst-facing output engineering**.

## 3.0 Imports & Runtime Setup

This first cell imports all runtime dependencies used in the notebook.

Broadly, the imports fall into these categories:

- **data handling**: `pandas`, `numpy`, `ast`, `json`
- **HTTP integration**: `requests`
- **timing and retries**: `time`, `datetime`
- **prompt assembly and formatting**: `textwrap`
- **visualization**: `matplotlib`, `plotly`
- **progress tracking**: `tqdm`

The notebook also adjusts Pandas display settings to make inspection cells more readable.

In [ ]:
import os, ast, json, time, warnings, textwrap
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
from datetime import datetime
from typing import Optional
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)
print('✅ Imports loaded.')


## 3.1 Load upstream artifacts

This section loads the outputs of Notebook 02.

### Required input
- `flagged_users.csv`

### Optional input
- `risk_scores.csv`

The optional history file allows the notebook to add **recent-vs-baseline trend context** to the prompt.

### Primary alert candidates

The next cell loads the condensed flagged-user file generated by Notebook 02.

This dataset is already a triaged view of the full scored matrix and typically contains:

- one row per flagged user,
- the user's peak-risk day,
- model scores,
- SHAP top-5 context,
- selected behavioral feature values.


In [ ]:
FLAGGED_PATH = 'flagged_users.csv'
if not os.path.exists(FLAGGED_PATH):
    raise FileNotFoundError(f'❌ {FLAGGED_PATH} not found. Please run Notebook 02 first.')
flagged = pd.read_csv(FLAGGED_PATH, parse_dates=['date_only'])
flagged = flagged.sort_values('risk_score', ascending=False).reset_index(drop=True)
print(f'✅ Flagged users loaded: {len(flagged):,} records')
print(flagged['alert_level'].value_counts().to_string())
flagged[['user_id','date_only','risk_score','alert_level','iso_score','ae_score','lstm_score']].head(5)


### Optional risk history for trend context

The next cell loads the full scored user-day dataset if available.

This is used later to compute a lightweight trend signal:

- 7-day average risk,
- 30-day average risk,
- short-term delta.

That trend summary helps the LLM distinguish a one-off spike from a building pattern.

In [ ]:
RISK_SCORES_PATH = 'risk_scores.csv'
if os.path.exists(RISK_SCORES_PATH):
    risk_scores = pd.read_csv(RISK_SCORES_PATH, parse_dates=['date_only'])
    print(f'✅ risk_scores.csv loaded: {risk_scores.shape}')
else:
    risk_scores = None
    print('⚠️  risk_scores.csv not found — baseline delta context will be skipped.')


## 3.2 PCAI endpoint configuration

This section defines the runtime settings used to call the external LLM.

Parameters include:

- endpoint URL,
- API key,
- model name,
- decoding settings,
- timeout,
- retry policy.

### Important operational note
This notebook currently embeds credentials directly in code and disables TLS verification later. That may be acceptable in a tightly controlled lab, but it is **not production-safe**. For real deployments, secrets should come from environment variables or a secret store.

In [ ]:
PCAI_ENDPOINT   = "https://do-not-delete--gpt-oss-120b.project-user-hugo.serving.ai-application.pcai.hpecic.net/v1/chat/completions"
PCAI_API_KEY    = os.getenv('PCAI_API_KEY', 'REPLACE_WITH_API_KEY')
PCAI_MODEL      = "RedHatAI/gpt-oss-120b"
LLM_TEMPERATURE = 0.1
LLM_MAX_TOKENS  = 2000
LLM_TOP_P       = 0.9
LLM_TIMEOUT_SEC = 60
BATCH_DELAY_SEC = 0.5
MAX_RETRIES     = 3
RETRY_DELAY_SEC = 5
print('✅ PCAI configuration set.')
print(f'   Endpoint : {PCAI_ENDPOINT}')
print(f'   Model    : {PCAI_MODEL}')


## 3.3 LLM client wrapper

This section defines the reusable HTTP client used to call the PCAI endpoint.

### Design goals
- keep payload construction centralized,
- add retry behavior,
- handle timeouts and HTTP failures,
- return only the assistant content field.

This wrapper is the notebook's integration boundary with the external model service.

### TLS handling note

The next code cell disables TLS warnings, and the client later uses `verify=False`.

That keeps lab execution friction low, but it is a genuine security trade-off. In production or enterprise testing, certificate validation should be restored.

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def call_llm(
    system_prompt: str,
    user_prompt: str,
    temperature: float = LLM_TEMPERATURE,
    max_tokens: int = LLM_MAX_TOKENS,
    retries: int = MAX_RETRIES,
) -> Optional[str]:
    headers = {
        'Authorization': f'Bearer {PCAI_API_KEY}',
        'Content-Type' : 'application/json',
    }
    payload = {
        'model': PCAI_MODEL, 'temperature': temperature,
        'max_tokens': max_tokens, 'top_p': LLM_TOP_P,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                PCAI_ENDPOINT, headers=headers, json=payload,
                timeout=LLM_TIMEOUT_SEC, verify=False,
            )
            response.raise_for_status()
            return response.json()['choices'][0]['message']['content']
        except requests.exceptions.Timeout:
            print(f'   ⚠️  Attempt {attempt}/{retries}: Timeout after {LLM_TIMEOUT_SEC}s')
        except requests.exceptions.HTTPError as e:
            print(f'   ⚠️  Attempt {attempt}/{retries}: HTTP {e.response.status_code}')
            if e.response.status_code in [400, 401, 403]:
                print('   ❌ Non-retryable error. Check endpoint/API key.')
                return None
        except Exception as e:
            print(f'   ⚠️  Attempt {attempt}/{retries}: {type(e).__name__}: {e}')
        if attempt < retries:
            time.sleep(RETRY_DELAY_SEC)
    print(f'   ❌ All {retries} attempts failed.')
    return None

print('✅ LLM client function defined (TLS verification disabled).')


## 3.4 Connectivity test

Before building prompts for real alerts, the notebook performs a very small health check against the model endpoint.

This is good practice because it separates:

- **integration failures**,
- from **prompting / parsing failures**.

If this step fails, there is no point continuing further down the notebook.

In [ ]:
print('🔌 Testing PCAI LLM connection...\n')
test_response = call_llm(
    system_prompt='You are a helpful assistant. Reply in one sentence.',
    user_prompt='Confirm you are online and ready for UEBA alert generation.',
    max_tokens=60,
)
if test_response:
    print('✅ Connection successful!')
    print(f'   Model response: {test_response.strip()}')
else:
    print('❌ Connection failed. Check PCAI_ENDPOINT and PCAI_API_KEY.')


## 3.5 System prompt design

This system prompt defines the model's **role**, **constraints**, and **output contract**.

### Key prompt design choices
- explicitly frames the model as a **senior cybersecurity threat analyst**,
- tells it to avoid inventing facts,
- requires consideration of benign explanations,
- forces output to be a **single valid JSON object**.

This is a strong pattern for operational LLM systems because it pushes structure and caution directly into the instruction layer.

### Required response schema

The notebook expects this JSON structure:

| **Field** | **Purpose** |
|---|---|
| `threat_type` | suspected threat category |
| `confidence` | confidence label |
| `summary` | concise analyst-readable narrative |
| `key_evidence` | core evidence bullets |
| `recommended_action` | next action for SOC |
| `false_positive_risk` | false-positive likelihood |
| `false_positive_reason` | why the activity may be benign |
| `kill_chain_stage` | approximate attack stage |

This schema is compact enough for repeated batch use and rich enough to support dashboards or ticketing pipelines.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are a senior cybersecurity threat analyst specialising in
    User and Entity Behavior Analytics (UEBA) and insider threat detection.

    Your task is to analyse ML-generated anomaly evidence for a flagged
    employee and produce a structured SOC alert in valid JSON format.

    Guidelines:
    - Reason carefully from the evidence provided. Do not invent facts.
    - Consider benign explanations (travel, project deadlines, role changes)
      before concluding malicious intent.
    - Classify the most likely threat type from the evidence.
    - Assess false positive risk honestly.
    - Keep the summary concise and actionable for a SOC analyst.
    - Your entire response must be a single valid JSON object.
      Do not include markdown fences, commentary, or any text outside the JSON.

    Required JSON fields:
    {
      \"threat_type\"           : string,
      \"confidence\"            : string,
      \"summary\"               : string,
      \"key_evidence\"          : list of strings,
      \"recommended_action\"    : string,
      \"false_positive_risk\"   : string,
      \"false_positive_reason\" : string,
      \"kill_chain_stage\"      : string
    }
""").strip()
print('✅ System prompt defined.')
print(f'   Length: {len(SYSTEM_PROMPT)} characters')


## 3.6 Evidence context builder

This is one of the most important parts of the notebook.

The context builder transforms a flagged row into a structured evidence package with sections such as:

- employee profile,
- ML model scores,
- alert history,
- SHAP top drivers,
- behavioral snapshot,
- optional risk trend.

### Why this matters
The quality of LLM outputs is heavily driven by the quality and structure of the evidence it receives. This notebook wisely does **not** send a raw CSV row and hope for brilliance.

### Context anatomy

The generated context is organized like a mini case file:

```text
=== EMPLOYEE PROFILE ===
=== ML ANOMALY SCORES ===
=== ALERT HISTORY ===
=== TOP ANOMALY DRIVERS (SHAP) ===
=== BEHAVIORAL FEATURE SNAPSHOT ===
=== RISK TREND ===
```

That structure helps the LLM reason over the evidence in a stable, repeatable way.

In [ ]:
def build_context(row: pd.Series, risk_scores_df: Optional[pd.DataFrame] = None) -> str:
    lines = []
    lines.append('=== EMPLOYEE PROFILE ===')
    lines.append(f'User ID     : {row["user_id"]}')
    if 'role' in row and pd.notna(row.get('role')):
        lines.append(f'Role        : {row["role"]}')
    if 'department' in row and pd.notna(row.get('department')):
        lines.append(f'Department  : {row["department"]}')
    lines.append(f'Peak Risk Day: {str(row["date_only"])[:10]}')
    lines.append('')
    lines.append('=== ML ANOMALY SCORES (0=normal, 1=anomalous) ===')
    lines.append(f'Ensemble Risk Score : {row["risk_score"]:.4f}  [{row["alert_level"].upper()}]')
    lines.append(f'Isolation Forest    : {row["iso_score"]:.4f}')
    lines.append(f'Autoencoder         : {row["ae_score"]:.4f}')
    lines.append(f'LSTM Sequence       : {row["lstm_score"]:.4f}')
    lines.append('')
    lines.append('=== ALERT HISTORY (last 30 days) ===')
    for level in ['critical_days', 'elevated_days', 'watch_days']:
        if level in row and pd.notna(row.get(level)):
            lines.append(f'{level.replace("_days","").upper():<12}: {int(row[level])} day(s)')
    lines.append('')
    lines.append('=== TOP ANOMALY DRIVERS (SHAP) ===')
    if 'shap_top5' in row and pd.notna(row.get('shap_top5')):
        try:
            shap_list = ast.literal_eval(str(row['shap_top5']))
            for i, item in enumerate(shap_list, 1):
                direction = 'raises risk' if float(item.get('shap_value', 0)) > 0 else 'lowers risk'
                lines.append(f'{i}. {item["feature"]:<35} value={item["feat_value"]:>8.4f}  shap={item["shap_value"]:>8.5f}  [{direction}]')
        except Exception:
            lines.append(f'   {str(row["shap_top5"])[:300]}')
    else:
        lines.append('   (SHAP data not available)')
    lines.append('')
    lines.append('=== BEHAVIORAL FEATURE SNAPSHOT (peak risk day) ===')
    for feat in ['files_accessed','sensitive_file_ratio','delete_ratio','after_hours_ratio',
                 'usb_event_count','usb_bytes_gb','email_count','external_email_ratio',
                 'emails_with_attachments','job_site_visits','cloud_storage_visits',
                 'webmail_visits','login_count','unique_pcs','session_span_hours']:
        if feat in row and pd.notna(row.get(feat)):
            lines.append(f'  {feat:<35}: {float(row[feat]):>10.4f}')
    if risk_scores_df is not None:
        user_history = risk_scores_df[risk_scores_df['user_id'] == row['user_id']].sort_values('date_only')
        if len(user_history) >= 7:
            recent_7d    = user_history.tail(7)['risk_score'].mean()
            baseline_30d = user_history.tail(30)['risk_score'].mean()
            delta = recent_7d - baseline_30d
            trend = 'INCREASING \u2191' if delta > 0.05 else 'DECREASING \u2193' if delta < -0.05 else 'STABLE \u2192'
            lines += ['', '=== RISK TREND ===',
                      f'7-day avg risk   : {recent_7d:.4f}',
                      f'30-day avg risk  : {baseline_30d:.4f}',
                      f'Delta            : {delta:+.4f}  [{trend}]']
    return '\n'.join(lines)

print('✅ Context builder defined.')
sample_context = build_context(flagged.iloc[0], risk_scores)
print(sample_context)


## 3.7 User prompt builder

This prompt template wraps the evidence in a step-by-step reasoning scaffold.

The LLM is explicitly asked to walk through:

1. evidence review,
2. benign explanation,
3. threat hypothesis,
4. confidence assessment,
5. final JSON output.

This kind of structured reasoning prompt is very helpful when you want the final answer to be disciplined without exposing or depending on free-form chain-of-thought output.

### Prompting pattern used

The design here follows a good operational template:

```text
Role instruction (system)
   +
Structured reasoning scaffold (user)
   +
Structured evidence block
   +
Strict output format requirement
```

That combination is usually more reliable than simply saying, “Please analyze this suspicious user.”

In [ ]:
def build_user_prompt(context: str) -> str:
    return textwrap.dedent(f"""
        Below is the ML-generated anomaly evidence for a flagged employee.
        Analyse this evidence carefully using the following reasoning steps:

        STEP 1 - EVIDENCE REVIEW
        Identify which behavioral signals are most significant and why.

        STEP 2 - BENIGN HYPOTHESIS
        Consider whether legitimate business activities (e.g., project
        deadlines, role changes, travel, IT tasks) could explain the anomalies.

        STEP 3 - THREAT HYPOTHESIS
        If the evidence suggests malicious intent, identify the most likely
        threat type and kill chain stage.

        STEP 4 - CONFIDENCE ASSESSMENT
        Weigh both hypotheses and assign confidence and false positive risk.

        STEP 5 - JSON OUTPUT
        Produce ONLY the final JSON object. No preamble, no markdown fences.

        --- EVIDENCE ---
        {context}
        --- END EVIDENCE ---

        Respond with the JSON alert object now:
    """).strip()

print('✅ Chain-of-thought prompt template defined.')
print(f'Sample prompt length: {len(build_user_prompt(sample_context))} characters')


## 3.8 Response parsing and validation

This section makes the notebook robust to common LLM output issues.

### Problems it guards against
- missing response,
- stray markdown fences,
- extra commentary around JSON,
- truncated output,
- malformed JSON,
- missing required fields,
- wrong datatypes.

This is a very important engineering step. Without response validation, LLM pipelines tend to fail in creative and inconvenient ways.

### Parse status semantics

The parser assigns one of three statuses:

- `ok`      → valid JSON with expected structure,
- `partial` → JSON parsed but some required fields were missing or normalized,
- `failed`  → no safe structured alert could be recovered.

This gives downstream code a simple quality signal without crashing the whole batch.

In [ ]:
REQUIRED_FIELDS = ['threat_type','confidence','summary','key_evidence',
                   'recommended_action','false_positive_risk','false_positive_reason','kill_chain_stage']
VALID_CONFIDENCE = ['High', 'Medium', 'Low']
VALID_FP_RISK    = ['High', 'Medium', 'Low']

def parse_llm_response(raw: str, user_id: str) -> dict:
    if raw is None:
        return {'user_id': user_id, 'parse_status': 'failed', 'parse_error': 'LLM returned None'}
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        cleaned = '\n'.join(l for l in cleaned.split('\n') if not l.strip().startswith('```')).strip()
    start = cleaned.find('{')
    end   = cleaned.rfind('}') + 1
    if start == -1 or end == 0:
        if start != -1 and cleaned[start:].count('{') > cleaned[start:].count('}'):
            return {'user_id': user_id, 'parse_status': 'failed',
                    'parse_error': 'Truncated JSON — increase LLM_MAX_TOKENS',
                    'raw_response': cleaned[:500]}
        return {'user_id': user_id, 'parse_status': 'failed',
                'parse_error': 'No JSON object found', 'raw_response': cleaned[:500]}
    try:
        alert = json.loads(cleaned[start:end])
    except json.JSONDecodeError as e:
        open_braces  = cleaned[start:].count('{')
        close_braces = cleaned[start:].count('}')
        if open_braces > close_braces:
            parse_error = f'Truncated JSON (unclosed braces: +{open_braces - close_braces}) — increase LLM_MAX_TOKENS'
        else:
            parse_error = str(e)
        return {'user_id': user_id, 'parse_status': 'failed',
                'parse_error': parse_error, 'raw_response': cleaned[:500]}
    alert['user_id'] = user_id
    issues = []
    for field in REQUIRED_FIELDS:
        if field not in alert:
            alert[field] = 'Unknown'
            issues.append(f'missing: {field}')
    for field, valid_set in [('confidence', VALID_CONFIDENCE), ('false_positive_risk', VALID_FP_RISK)]:
        val = str(alert.get(field, '')).strip().title()
        alert[field] = val if val in valid_set else 'Unknown'
    if not isinstance(alert.get('key_evidence'), list):
        alert['key_evidence'] = [str(alert.get('key_evidence', ''))]
    alert['parse_status'] = 'partial' if issues else 'ok'
    if issues:
        alert['parse_issues'] = issues
    return alert

print('✅ JSON response parser defined.')


## 3.9 Single-user dry run

Before running a batch across many users, the notebook performs a single-user test on the top-risk case.

This is excellent notebook hygiene because it lets you inspect:

- prompt quality,
- endpoint health,
- parse behavior,
- schema conformance,
- overall alert style.


### What to inspect in the dry run

For the output below, a developer should look at:

- whether the summary sticks to provided evidence,
- whether benign alternatives are considered,
- whether JSON parses cleanly,
- whether field values match the allowed schema.


In [ ]:
print('🧪 Running single-user alert test on top-risk user...\n')
test_row     = flagged.iloc[0]
test_context = build_context(test_row, risk_scores)
test_prompt  = build_user_prompt(test_context)
print(f'   User ID     : {test_row["user_id"]}')
print(f'   Risk score  : {test_row["risk_score"]:.4f}')
print(f'   Alert level : {test_row["alert_level"].upper()}')
print(f'   Prompt len  : {len(test_prompt)} chars')
print('\n   Calling PCAI LLM...')
test_raw   = call_llm(SYSTEM_PROMPT, test_prompt)
test_alert = parse_llm_response(test_raw, test_row['user_id'])
print(f'\n✅ Test alert generated. Parse status: {test_alert["parse_status"].upper()}')
print('\n📋 Alert output:')
print(json.dumps(test_alert, indent=2))


## 3.10 Batch planning

This section determines how many users to process and in what order.

### Prioritization policy
Users are sorted by:

1. alert severity (`critical` first),
2. then descending `risk_score`.

This ensures scarce LLM time is spent first on the most operationally important cases.

### Why `MAX_USERS` exists

`MAX_USERS` is a practical safety control for workshop runs:

- limits API usage,
- keeps run time bounded,
- makes iterative prompt tuning easier.

During development, keeping a small cap is often the difference between fast iteration and accidental coffee breaks.

In [ ]:
MAX_USERS = 20
priority_order = {'critical': 0, 'elevated': 1, 'watch': 2}
flagged_sorted = flagged.copy()
flagged_sorted['priority'] = flagged_sorted['alert_level'].map(priority_order).fillna(3)
flagged_sorted = flagged_sorted.sort_values(
    ['priority', 'risk_score'], ascending=[True, False]
).reset_index(drop=True)
if MAX_USERS is not None:
    batch = flagged_sorted.head(MAX_USERS)
    print(f'⚠️  MAX_USERS={MAX_USERS}: processing top {MAX_USERS} users only.')
    print('   Set MAX_USERS = None to process all flagged users.')
else:
    batch = flagged_sorted
print(f'\n✅ Batch ready: {len(batch)} users to process')
print(batch['alert_level'].value_counts().to_string())
est_minutes = len(batch) * (LLM_TIMEOUT_SEC / 60 * 0.3 + BATCH_DELAY_SEC / 60)
print(f'\n   Estimated time: ~{est_minutes:.1f} minutes')


## 3.11 Batch alert generation

This is the main execution loop of the notebook.

For each flagged user, it performs:

1. context building,
2. prompt generation,
3. LLM call,
4. JSON parsing,
5. metadata augmentation,
6. result collection.

The loop also tracks failures separately so the notebook can finish gracefully even if some alerts do not parse.

### Batch execution pattern

The loop follows a strong systems pattern:

```text
for each candidate:
   build deterministic evidence
   call probabilistic model
   normalize into deterministic schema
   store enriched result
```

That pattern is often the difference between a useful LLM notebook and a pile of very eloquent chaos.

In [ ]:
alerts     = []
failed_ids = []
start_time = time.time()
print(f'🚀 Starting batch alert generation for {len(batch)} users...\n')
for idx, row in tqdm(batch.iterrows(), total=len(batch), desc='Generating alerts'):
    user_id      = row['user_id']
    context      = build_context(row, risk_scores)
    user_prompt  = build_user_prompt(context)
    raw_response = call_llm(SYSTEM_PROMPT, user_prompt)
    alert        = parse_llm_response(raw_response, user_id)
    alert['risk_score']    = float(row['risk_score'])
    alert['alert_level']   = str(row['alert_level'])
    alert['iso_score']     = float(row['iso_score'])
    alert['ae_score']      = float(row['ae_score'])
    alert['lstm_score']    = float(row['lstm_score'])
    alert['peak_risk_day'] = str(row['date_only'])[:10]
    alert['generated_at']  = datetime.utcnow().isoformat() + 'Z'
    if 'role' in row and pd.notna(row.get('role')):
        alert['role'] = str(row['role'])
    if 'department' in row and pd.notna(row.get('department')):
        alert['department'] = str(row['department'])
    alerts.append(alert)
    if alert['parse_status'] == 'failed':
        failed_ids.append(user_id)
    time.sleep(BATCH_DELAY_SEC)
elapsed       = time.time() - start_time
ok_count      = sum(1 for a in alerts if a['parse_status'] == 'ok')
partial_count = sum(1 for a in alerts if a['parse_status'] == 'partial')
failed_count  = sum(1 for a in alerts if a['parse_status'] == 'failed')
print(f'\n✅ Batch complete in {elapsed:.1f}s ({elapsed/max(len(batch),1):.1f}s per user)')
print(f'   OK      : {ok_count}')
print(f'   Partial : {partial_count}')
print(f'   Failed  : {failed_count}')
if failed_ids:
    print(f'   Failed users: {failed_ids}')


## 3.12 Human-readable alert preview

This preview cell prints the first few generated alerts in a compact console-friendly format.

Even though the saved outputs are structured, this preview is useful for rapid developer sanity checks on:

- tone,
- consistency,
- evidence relevance,
- recommendation quality.


In [ ]:
print('📋 Preview - first 3 generated alerts:\n')
for alert in alerts[:3]:
    print('=' * 65)
    print(f'  User         : {alert.get("user_id")}')
    print(f'  Risk Score   : {alert.get("risk_score", 0):.4f}  [{alert.get("alert_level", "?").upper()}]')
    print(f'  Threat Type  : {alert.get("threat_type", "?")}')
    print(f'  Confidence   : {alert.get("confidence", "?")}')
    print(f'  Kill Chain   : {alert.get("kill_chain_stage", "?")}')
    print(f'  FP Risk      : {alert.get("false_positive_risk", "?")}')
    print(f'  Parse Status : {alert.get("parse_status", "?").upper()}')
    print('  Summary      :')
    for line in textwrap.wrap(alert.get('summary', ''), width=60):
        print(f'    {line}')
    print('  Key Evidence :')
    for ev in alert.get('key_evidence', [])[:3]:
        for line in textwrap.wrap(str(ev), width=58):
            print(f'    - {line}')
    print('  Recommended  :')
    for line in textwrap.wrap(alert.get('recommended_action', ''), width=60):
        print(f'    {line}')
print('=' * 65)


## 3.13 Alert DataFrame construction

This section materializes the generated alerts into Pandas DataFrames for downstream analysis and visualization.

Two views are created:

- `alerts_df`: all generated alerts,
- `alerts_valid`: only alerts whose parse status is not `failed`.

This separation is useful because even if some responses fail, the notebook can still analyze the successful subset.

In [ ]:
alerts_df    = pd.DataFrame(alerts)
alerts_valid = alerts_df[alerts_df['parse_status'] != 'failed'].copy() \
               if 'parse_status' in alerts_df.columns else alerts_df.copy()
print(f'✅ Alerts DataFrame built: {alerts_df.shape}')
print(f'   Valid alerts for analysis: {len(alerts_valid)}')
print()

def print_dist(df: pd.DataFrame, col: str, label: str) -> None:
    print(f'{label}:')
    if df.empty:
        print('   (no valid alerts - all LLM parses failed)')
    elif col not in df.columns:
        print(f'   (column "{col}" not present - check parse_llm_response)')
    else:
        print(df[col].value_counts().to_string())
    print()

print_dist(alerts_valid, 'threat_type',        'Threat type distribution')
print_dist(alerts_valid, 'confidence',          'Confidence distribution')
print_dist(alerts_valid, 'false_positive_risk', 'False positive risk distribution')


## 3.14 Alert quality visualization

This dashboard gives a quick operational readout of the generated LLM alerts.

It visualizes:

- threat type distribution,
- LLM confidence distribution,
- false-positive risk distribution,
- relationship between ML risk and LLM confidence.

The final panel is especially useful because it reminds us that:

- a high ML score does **not automatically** imply high narrative confidence,
- contextual ambiguity can still make the LLM cautious.

In [ ]:
if alerts_valid.empty:
    print('⚠️  No valid alerts to visualise. All LLM responses failed to parse.')
else:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Threat Type Distribution', 'LLM Confidence Distribution',
            'False Positive Risk Distribution', 'ML Risk Score vs LLM Confidence',
        ],
        specs=[
            [{'type': 'xy'},     {'type': 'domain'}],
            [{'type': 'domain'}, {'type': 'xy'}    ],
        ],
    )
    threat_colors = {
        'Data Exfiltration': '#FF4444', 'Sabotage':       '#FF6B35',
        'Privilege Abuse':   '#FFD700', 'Reconnaissance': '#DA70D6',
        'Benign':            '#00C853', 'Unknown':        '#888888',
    }
    threat_counts = alerts_valid['threat_type'].value_counts()
    fig.add_trace(go.Bar(
        x=threat_counts.index.tolist(), y=threat_counts.values.tolist(),
        marker_color=[threat_colors.get(t, '#888') for t in threat_counts.index],
        name='Threat Type',
    ), row=1, col=1)
    conf_counts = alerts_valid['confidence'].value_counts()
    conf_colors = {'High': '#FF4444', 'Medium': '#FFD700', 'Low': '#00C853', 'Unknown': '#888'}
    fig.add_trace(go.Pie(
        labels=conf_counts.index.tolist(), values=conf_counts.values.tolist(),
        marker_colors=[conf_colors.get(c, '#888') for c in conf_counts.index],
        hole=0.4, name='Confidence',
    ), row=1, col=2)
    fp_counts = alerts_valid['false_positive_risk'].value_counts()
    fp_colors = {'High': '#00C853', 'Medium': '#FFD700', 'Low': '#FF4444', 'Unknown': '#888'}
    fig.add_trace(go.Pie(
        labels=fp_counts.index.tolist(), values=fp_counts.values.tolist(),
        marker_colors=[fp_colors.get(f, '#888') for f in fp_counts.index],
        hole=0.4, name='FP Risk',
    ), row=2, col=1)
    conf_numeric = {'High': 3, 'Medium': 2, 'Low': 1, 'Unknown': 0}
    alerts_valid = alerts_valid.copy()
    alerts_valid['conf_num'] = alerts_valid['confidence'].map(conf_numeric).fillna(0)
    fig.add_trace(go.Scatter(
        x=alerts_valid['risk_score'],
        y=alerts_valid['conf_num'] + np.random.uniform(-0.1, 0.1, len(alerts_valid)),
        mode='markers',
        marker=dict(size=8, color=[threat_colors.get(t, '#888') for t in alerts_valid['threat_type']], opacity=0.8),
        text=alerts_valid['user_id'], name='Users',
    ), row=2, col=2)
    fig.update_yaxes(tickvals=[1,2,3], ticktext=['Low','Medium','High'], row=2, col=2)
    fig.update_xaxes(title_text='ML Risk Score',  row=2, col=2)
    fig.update_yaxes(title_text='LLM Confidence', row=2, col=2)
    fig.update_layout(
        height=700, plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
        font_color='white', showlegend=False,
        title_text='LLM Alert Quality Review', title_font_size=14,
    )
    fig.show()
    print('✅ Alert quality review dashboard rendered.')


## 3.15 False-positive filtering

This section applies a simple policy to split valid alerts into:

- likely benign / low-priority items,
- actionable alerts.

### Current filtering rule
An alert is filtered out if:

- `threat_type == 'Benign'`, or
- `false_positive_risk == 'High'`.

This is intentionally simple for workshop use. In a production system, you would likely use a more nuanced triage taxonomy such as:

- `actionable`,
- `review`,
- `likely benign`.


In [ ]:
if alerts_valid.empty:
    fp_filtered = alerts_valid.copy()
    true_alerts = alerts_valid.copy()
    print('⚠️  No valid alerts - FP filtering skipped.')
else:
    fp_filter_mask = (
        (alerts_valid['threat_type'] == 'Benign') |
        (alerts_valid['false_positive_risk'] == 'High')
    )
    fp_filtered = alerts_valid[fp_filter_mask].copy()
    true_alerts = alerts_valid[~fp_filter_mask].copy()
    n_valid = max(len(alerts_valid), 1)
    print('=' * 60)
    print('  FALSE POSITIVE FILTERING SUMMARY')
    print('=' * 60)
    print(f'  Total valid alerts    : {len(alerts_valid):>5}')
    print(f'  LLM-filtered as FP    : {len(fp_filtered):>5} ({len(fp_filtered)/n_valid*100:.1f}%)')
    print(f'  Actionable alerts     : {len(true_alerts):>5} ({len(true_alerts)/n_valid*100:.1f}%)')
    print('=' * 60)
    if len(fp_filtered) > 0:
        print('\n  Sample filtered (likely benign) alerts:')
        for _, row in fp_filtered.head(3).iterrows():
            print(f'  - {row["user_id"]} | risk={row["risk_score"]:.4f} | '
                  f'FP reason: {str(row.get("false_positive_reason", ""))[:80]}')


## 3.16 ML vs LLM comparison view

This table is a useful reconciliation step between the anomaly models and the reasoning model.

It helps answer:

- which high-risk ML alerts remain actionable after contextual reasoning,
- whether LLM confidence tracks ML risk,
- which kill-chain stages are being inferred from the evidence.

In [ ]:
print('📊 ML Score vs LLM Verdict - Top 15 Actionable Alerts:\n')
if true_alerts.empty:
    print('⚠️  No actionable alerts to display.')
else:
    comparison_cols = ['user_id','risk_score','alert_level',
                       'threat_type','confidence','false_positive_risk','kill_chain_stage']
    comparison_cols = [c for c in comparison_cols if c in true_alerts.columns]
    comparison = (
        true_alerts[comparison_cols]
        .sort_values('risk_score', ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    comparison.index += 1
    print(comparison.to_string())
    print('\n💡 Key insight: LLM confidence does not always track ML risk score.')
    print('   High ML score + Low LLM confidence  = investigate context further.')
    print('   High ML score + High LLM confidence = prioritise for SOC action.')


## 3.17 Kill chain distribution

The next chart summarizes inferred kill-chain stages for the actionable alert subset.

This is a lightweight way to understand what type of analyst workload the model is generating. For example:

- are most alerts being framed as recon or collection?
- are alerts concentrated in exfiltration-like activity?
- is the taxonomy stable enough for dashboarding?


In [ ]:
if true_alerts.empty or 'kill_chain_stage' not in true_alerts.columns:
    print('⚠️  No actionable alerts - kill chain chart skipped.')
else:
    kc_counts = true_alerts['kill_chain_stage'].value_counts().reset_index()
    kc_counts.columns = ['stage', 'count']
    kc_order = ['Reconnaissance','Weaponization','Delivery','Exploitation',
                'Installation','C2','Exfiltration','Unknown']
    kc_counts['order'] = kc_counts['stage'].apply(
        lambda s: kc_order.index(s) if s in kc_order else 99
    )
    kc_counts = kc_counts.sort_values('order')
    fig = px.bar(
        kc_counts, x='stage', y='count',
        title='Kill Chain Stage Distribution - Actionable Alerts',
        labels={'stage': 'Kill Chain Stage', 'count': 'Alert Count'},
        color='count', color_continuous_scale='Reds', height=380,
    )
    fig.update_layout(
        plot_bgcolor='#0D1B2A', paper_bgcolor='#0D1B2A',
        font_color='white', title_font_size=13,
        coloraxis_showscale=False, xaxis_tickangle=-20,
    )
    fig.show()
    print('✅ Kill chain distribution rendered.')


## 3.18 Persist structured outputs

This section writes two artifacts:

| **File** | **Purpose** |
|---|---|
| `alerts.json` | full structured output including metadata and all alerts |
| `alerts_summary.csv` | flattened analyst-friendly summary table |

These files are designed to be consumed by dashboards, reporting notebooks, or validation steps in Notebook 04.

### JSON artifact

`alerts.json` preserves the richest form of the output, including:

- generation metadata,
- model identifier,
- batch statistics,
- full alert objects.

This is the best export for machine-to-machine consumption.

In [ ]:
ALERTS_JSON_PATH = 'alerts.json'
output = {
    'generated_at'     : datetime.utcnow().isoformat() + 'Z',
    'model'            : PCAI_MODEL,
    'total_alerts'     : len(alerts),
    'ok_count'         : ok_count,
    'partial_count'    : partial_count,
    'failed_count'     : failed_count,
    'fp_filtered_count': len(fp_filtered),
    'actionable_count' : len(true_alerts),
    'alerts'           : alerts,
}
with open(ALERTS_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, default=str)
print('✅ alerts.json saved.')
print(f'   Path  : {ALERTS_JSON_PATH}')
print(f'   Size  : {os.path.getsize(ALERTS_JSON_PATH)/1e3:.1f} KB')
print(f'   Total alerts in file: {len(alerts)}')


### CSV summary artifact

`alerts_summary.csv` provides a flatter, human-readable export for:

- quick review,
- spreadsheets,
- dashboards,
- downstream validation notebooks.


In [ ]:
ALERTS_CSV_PATH = 'alerts_summary.csv'
summary_cols = [
    'user_id', 'peak_risk_day', 'risk_score', 'alert_level',
    'iso_score', 'ae_score', 'lstm_score',
    'threat_type', 'confidence', 'kill_chain_stage',
    'false_positive_risk', 'false_positive_reason',
    'summary', 'recommended_action', 'parse_status', 'generated_at',
]
if 'role' in alerts_df.columns:
    summary_cols.insert(2, 'role')
if 'department' in alerts_df.columns:
    summary_cols.insert(3, 'department')
summary_cols = [c for c in summary_cols if c in alerts_df.columns]
alerts_df[summary_cols].to_csv(ALERTS_CSV_PATH, index=False)
print('✅ alerts_summary.csv saved.')
print(f'   Path  : {ALERTS_CSV_PATH}')
print(f'   Shape : {alerts_df[summary_cols].shape}')
print(f'   Size  : {os.path.getsize(ALERTS_CSV_PATH)/1e3:.1f} KB')


## ✅ Final summary

The closing cell prints a compact end-of-run summary, including:

- model and endpoint identity,
- batch size,
- parse success metrics,
- false-positive filtering statistics,
- threat type breakdown,
- saved output locations.

This acts as the notebook's final execution health check.

In [ ]:
print('=' * 60)
print('  NOTEBOOK 03 - FINAL SUMMARY')
print('=' * 60)
print(f'  LLM endpoint      : {PCAI_ENDPOINT}')
print(f'  Model             : {PCAI_MODEL}')
print()
print(f'  Users processed   : {len(batch)}')
print(f'  Alerts generated  : {len(alerts)}')
print(f'    OK              : {ok_count}')
print(f'    Partial         : {partial_count}')
print(f'    Failed          : {failed_count}')
print()
n_valid = max(len(alerts_valid), 1)
print('  Alert quality:')
print(f'    FP filtered     : {len(fp_filtered)} ({len(fp_filtered)/n_valid*100:.1f}%)')
print(f'    Actionable      : {len(true_alerts)} ({len(true_alerts)/n_valid*100:.1f}%)')
print()
print('  Threat type breakdown:')
if alerts_valid.empty or 'threat_type' not in alerts_valid.columns:
    print('    (no valid alerts to summarise)')
else:
    for threat, count in alerts_valid['threat_type'].value_counts().items():
        print(f'    {threat:<22}: {count}')
print()
print(f'  Outputs saved:')
print(f'  -> {ALERTS_JSON_PATH}')
print(f'  -> {ALERTS_CSV_PATH}')
print('=' * 60)
print('\n🎯 Notebook 03 Complete!')
print('   -> Open 04_dashboard_and_validation.ipynb to build')
print('     the SOC dashboard and validate against ground truth.')


---
## 📌 Notebook 03 developer takeaways

### What this notebook adds to the pipeline
| **Capability** | **What it enables** |
|---|---|
| Structured context building | reliable evidence packaging for LLM calls |
| Strict JSON prompting | machine-readable analyst outputs |
| Parse validation | resilience to malformed model responses |
| False-positive assessment | lightweight alert suppression / triage |
| Export artifacts | downstream dashboards and validation |

### Architectural insight
This notebook demonstrates a strong pattern for LLM integration in security workflows:

- keep detection in classical/ML systems,
- use the LLM for summarization and reasoning,
- constrain outputs with schema and parser logic,
- preserve intermediate artifacts for auditability.

### Next step in the lab
Notebook 04 will consume the generated alert artifacts to build a dashboard and validate results against available ground truth.